## 0. Path configuration

Adjust `IMGS_DIR` and `SPLITS_DIR` according to your environment.

In [ ]:
import os
import json
import hashlib
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    BASE_DIR   = Path('/content/drive/MyDrive/Doutorado-V3/Trilha3-V2')
    IMGS_DIR   = Path('/content/drive/MyDrive/Doutorado-V3/Data/Imgs')
else:
    BASE_DIR   = Path('e:/Doutorado-V3/Trilha3-V2')
    IMGS_DIR   = Path('e:/Doutorado-V3/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'audit'
FIGS_DIR    = BASE_DIR / 'figs' / 'eda'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

CORE_LABELS  = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']

ARTIFACT_LABELS = ['SALIVA', 'LUZ']
N_FOLDS = 5

print('BASE_DIR  :', BASE_DIR)
print('IMGS_DIR  :', IMGS_DIR)
print('SPLITS_DIR:', SPLITS_DIR)
print('Existe?   ', SPLITS_DIR.exists())

## 1. Loading splits

In [ ]:
splits = {}
for fold in range(N_FOLDS):
    splits[fold] = {
        'train': pd.read_csv(SPLITS_DIR / f'fold_{fold}_train.csv'),
        'val':   pd.read_csv(SPLITS_DIR / f'fold_{fold}_val.csv'),
        'test':  pd.read_csv(SPLITS_DIR / f'fold_{fold}_test.csv'),
    }

print(f'{"Fold":<6} {"Train":>6} {"Val":>6} {"Test":>6} {"Total":>6}')
for fold in range(N_FOLDS):
    tr = len(splits[fold]['train'])
    vl = len(splits[fold]['val'])
    te = len(splits[fold]['test'])
    print(f'{fold:<6} {tr:>6} {vl:>6} {te:>6} {tr+vl+te:>6}')

print('\nColunas:', list(splits[0]['train'].columns))

## 2. Integrity audit — no data leakage between partitions

In [ ]:
leakage_report = {}

for fold in range(N_FOLDS):
    tr_ids  = set(splits[fold]['train']['image_name'])
    val_ids = set(splits[fold]['val']['image_name'])
    te_ids  = set(splits[fold]['test']['image_name'])

    tr_val  = tr_ids & val_ids
    tr_te   = tr_ids & te_ids
    val_te  = val_ids & te_ids

    leakage_report[f'fold_{fold}'] = {
        'train_val_overlap':  len(tr_val),
        'train_test_overlap': len(tr_te),
        'val_test_overlap':   len(val_te),
        'examples_train_val_overlap':  list(tr_val)[:5],
        'examples_train_test_overlap': list(tr_te)[:5],
    }

    status = '✓ OK' if (len(tr_val) + len(tr_te) + len(val_te)) == 0 else '✗ VAZAMENTO'
    print(f'Fold {fold}: train∩val={len(tr_val):>3}  train∩test={len(tr_te):>3}  val∩test={len(val_te):>3}  → {status}')

total_leakage = sum(
    v['train_val_overlap'] + v['train_test_overlap'] + v['val_test_overlap']
    for v in leakage_report.values()
)
print(f'\nVazamento total entre partições: {total_leakage}')

## 3. Disk file audit

In [ ]:
all_referenced = set()
for fold in range(N_FOLDS):
    for split in ['train', 'val', 'test']:
        all_referenced |= set(splits[fold][split]['image_name'])

    on_disk = set(p.name for p in IMGS_DIR.iterdir() if p.is_file() and p.suffix.lower() in ['.jpg', '.jpeg', '.png'])

missing      = all_referenced - on_disk
unreferenced = on_disk - all_referenced

file_report = {
    'total_referenced_unique': len(all_referenced),
    'total_on_disk':           len(on_disk),
    'missing_from_disk':       len(missing),
    'unreferenced_on_disk':    len(unreferenced),
    'missing_list':            sorted(missing)[:20],
}

print(f'Imagens únicas referenciadas nos splits : {len(all_referenced)}')
print(f'Arquivos em disco                       : {len(on_disk)}')
print(f'Faltando em disco                       : {len(missing)}')
print(f'Em disco mas não referenciadas          : {len(unreferenced)}')
if missing:
    print(f'Primeiras faltando: {sorted(missing)[:10]}')

## 4. Class distribution by fold and partition

In [ ]:
dist_report = {}

print(f'\n{"Classe":<22} ', end='')
for fold in range(N_FOLDS):
    print(f'F{fold}-Tr  F{fold}-Val F{fold}-Te  ', end='')
print()

for label in CORE_LABELS:
    dist_report[label] = {}
    print(f'{label:<22} ', end='')
    for fold in range(N_FOLDS):
        tr_n  = int(splits[fold]['train'][label].sum())
        val_n = int(splits[fold]['val'][label].sum())
        te_n  = int(splits[fold]['test'][label].sum())
        dist_report[label][f'fold_{fold}'] = {'train': tr_n, 'val': val_n, 'test': te_n}
        print(f'{tr_n:>5}  {val_n:>4}  {te_n:>4}  ', end='')
    print()

In [ ]:
print(f'\nImbalance Ratio (treino) — N_neg / N_pos por fold')
print(f'{"Classe":<22}', end='')
for fold in range(N_FOLDS):
    print(f'  Fold{fold}', end='')
print()

ir_report = {}
for label in CORE_LABELS:
    ir_report[label] = {}
    print(f'{label:<22}', end='')
    for fold in range(N_FOLDS):
        df_tr = splits[fold]['train']
        n_pos = df_tr[label].sum()
        n_neg = len(df_tr) - n_pos
        ir    = n_neg / n_pos if n_pos > 0 else float('inf')
        ir_report[label][f'fold_{fold}'] = round(ir, 1)
        print(f'  {ir:>5.1f}', end='')
    print()

In [ ]:
fig, axes = plt.subplots(1, len(CORE_LABELS), figsize=(14, 3.5), sharey=False)

for ax, label in zip(axes, CORE_LABELS):
    values = [dist_report[label][f'fold_{f}']['train'] for f in range(N_FOLDS)]
    bars = ax.bar(range(N_FOLDS), values, color='steelblue', edgecolor='white')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('Fold', fontsize=8)
    ax.set_xticks(range(N_FOLDS))
    ax.set_xticklabels([f'F{i}' for i in range(N_FOLDS)], fontsize=8)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(v), ha='center', va='bottom', fontsize=8)

axes[0].set_ylabel('Exemplos positivos (treino)', fontsize=9)
fig.suptitle('Positivos por classe e fold — conjunto de treino', fontsize=10, y=1.02)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'positivos_por_fold_treino.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 5. Artifact analysis × pathology (shortcut diagnosis)

In [ ]:
df_ref = pd.concat([splits[f]['train'] for f in range(N_FOLDS)]).drop_duplicates(subset=['image_name'])

shortcut_report = {}
print('Co-ocorrência artefato × patologia (fold-0 treino)\n')
print(f'{"Artefato":<8} {"Patologia":<22} {"N_patologia":>12} {"N_artefato+patologia":>22} {"% pat com artefato":>20}')
print('-' * 90)

for artifact in ARTIFACT_LABELS:
    for label in CORE_LABELS:
        n_pat       = int(df_ref[label].sum())
        n_co        = int(((df_ref[label] == 1) & (df_ref[artifact] == 1)).sum())
        pct         = 100 * n_co / n_pat if n_pat > 0 else 0
        shortcut_report[f'{artifact}+{label}'] = {
            'n_pathology': n_pat, 'n_cooccurrence': n_co, 'pct': round(pct, 1)
        }
        flag = ' ← ALTO RISCO' if pct > 20 else (' ← risco moderado' if pct > 10 else '')
        print(f'{artifact:<8} {label:<22} {n_pat:>12} {n_co:>22} {pct:>19.1f}%{flag}')

## 6. Multilabel subset — images with ≥2 positive labels

In [ ]:
multilabel_report = {}

for fold in range(N_FOLDS):
    for split_name in ['train', 'val', 'test']:
        df = splits[fold][split_name]
        n_total = len(df)
        n_multi = int((df[CORE_LABELS].sum(axis=1) >= 2).sum())
        n_single = int((df[CORE_LABELS].sum(axis=1) == 1).sum())
        n_zero   = int((df[CORE_LABELS].sum(axis=1) == 0).sum())
        multilabel_report[f'fold_{fold}_{split_name}'] = {
            'total': n_total,
            'zero_labels': n_zero,
            'single_label': n_single,
            'multi_label_ge2': n_multi,
            'pct_multi': round(100 * n_multi / n_total, 1)
        }

print(f'{"Fold":<6} {"Total":>6} {"0-rot":>6} {"1-rot":>6} {"≥2-rot":>7} {"% ≥2":>7}')
for fold in range(N_FOLDS):
    r = multilabel_report[f'fold_{fold}_train']
    print(f'{fold:<6} {r["total"]:>6} {r["zero_labels"]:>6} {r["single_label"]:>6} '
          f'{r["multi_label_ge2"]:>7} {r["pct_multi"]:>6.1f}%')

## 7. Expected pos_weight per class (reference for BCE)

In [ ]:
posweight_report = {}

print(f'{"Classe":<22}', end='')
for fold in range(N_FOLDS):
    print(f'  Fold{fold}', end='')
print('  Média')

for label in CORE_LABELS:
    posweight_report[label] = {}
    print(f'{label:<22}', end='')
    ws = []
    for fold in range(N_FOLDS):
        df_tr = splits[fold]['train']
        n_pos = df_tr[label].sum()
        n_neg = len(df_tr) - n_pos
        w = n_neg / n_pos if n_pos > 0 else 0
        posweight_report[label][f'fold_{fold}'] = round(w, 2)
        ws.append(w)
        print(f'  {w:>5.1f}', end='')
    print(f'  {np.mean(ws):>5.1f}')

## 8. SHA-256 of splits (reproducibility)

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            h.update(line.replace('\r\n', '\n').encode('utf-8'))
    return h.hexdigest()

hash_report = {}
for fold in range(N_FOLDS):
    for split_name in ['train', 'val', 'test']:
        fpath = SPLITS_DIR / f'fold_{fold}_{split_name}.csv'
        key   = f'fold_{fold}_{split_name}'
        hash_report[key] = sha256_file(fpath)
        print(f'{key:<25} {hash_report[key][:16]}...')

## 9. Save consolidated report

In [ ]:
report = {
    'metadata': {
        'notebook':   '00_auditoria_splits',
        'n_folds':    N_FOLDS,
        'core_labels': CORE_LABELS,
        'imgs_dir':   str(IMGS_DIR),
        'splits_dir': str(SPLITS_DIR),
    },
    'leakage':          leakage_report,
    'files':            file_report,
    'class_distribution': dist_report,
    'imbalance_ratio':  ir_report,
    'posweight':        posweight_report,
    'multilabel_dist':  multilabel_report,
    'shortcut_risk':    shortcut_report,
    'split_hashes':     hash_report,
    'summary': {
        'total_leakage':               total_leakage,
        'missing_images_from_disk':    file_report['missing_from_disk'],
        'splits_are_clean':            total_leakage == 0 and file_report['missing_from_disk'] == 0,
    }
}

out_path = RESULTS_DIR / 'audit_report.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f'Relatório salvo em: {out_path}')
print()
print('=== RESUMO FINAL ===')
print(f'  Vazamento entre partições : {total_leakage}')
print(f'  Imagens faltando em disco : {file_report["missing_from_disk"]}')
print(f'  Splits limpos?            : {report["summary"]["splits_are_clean"]}')

## 10. Figure — Imbalance Ratio per class (training, fold 0)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

mean_irs = [np.mean([ir_report[l][f'fold_{f}'] for f in range(N_FOLDS)]) for l in CORE_LABELS]
colors   = ['#d62728' if ir > 30 else '#ff7f0e' if ir > 10 else '#2ca02c' for ir in mean_irs]

bars = ax.barh(CORE_LABELS, mean_irs, color=colors, edgecolor='white')
for bar, v in zip(bars, mean_irs):
    ax.text(v + 0.5, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}', va='center', fontsize=9)

ax.axvline(10, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='IR=10 (moderado)')
ax.axvline(30, color='red',    linestyle='--', linewidth=1, alpha=0.7, label='IR=30 (severo)')
ax.set_xlabel('Imbalance Ratio médio (treino, 5 folds)')
ax.set_title('Desbalanceamento por classe — núcleo clínico', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'imbalance_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')